# CENG468 Veri Madenciliği · Hafta 5
## Kirli veriden ilk öneri listesine
Öğr. Gör. Dr. Duygu TOPALOĞLU · 2026-2027 Güz

---
### Bu derste nasıl çalışacağız?
Geçen haftaki işaretler aynen geçerli. İki yeni işaret ekledik:

| İşaret | Ne yapıyorsun? |
|---|---|
| 🔮 **Tahmin et** | Hücreyi çalıştırmadan önce çıktıyı tahmin et ve yaz. Sonra çalıştır ve karşılaştır. |
| 🔧 **Değiştir** | Çalışan kodu küçük bir farkla uyarla. |
| ✍️ **Kendin yaz** | Boş hücreye kodu kendin yaz. Kopyala-yapıştır yok. |
| 🐞 **Hata avı** | Kod bilerek hatalı. Hata mesajını oku, nedenini bul, düzelt. |
| 🧩 **Sırala** | Karışık verilen satırları doğru sıraya koyup çalıştır. |
| 🕵️ **İz sür** | Tuhaf bir sonucun nedenini veride bul. |

**İpuçları:** Bazı görevlerin altında kapalı 💡 kutular var. **Önce en az bir kez kendin dene**, takılırsan yalnızca bir sonraki ipucunu aç.

Bugünün yolu: **veriyi tanı → tuhaflıkları bul → temizlik kararı ver → kodla temizle → ilk öneri listesini çıkar.**

---
## Bölüm 0: Hazırlık (5 dk)
Bu defter kendi başına çalışır: veriyi kendisi okur. Önce aşağıdaki hücreyi çalıştır.

> İlk çalıştırmada Excel dosyası okunur (bilgisayara göre **30–90 saniye**) ve hızlı okunan bir CSV kopyası kaydedilir. Sonraki açılışlar birkaç saniye sürer. Hücrenin solunda `[*]` gördüğün sürece bekle.

In [ ]:
import os
import pandas as pd

# Veri klasörü (ders klasörüne göre). Geçen haftaki defterle aynı yer.
VERI = "Veri/online_retail"
XLSX = os.path.join(VERI, "online_retail_II.xlsx")
CSV = os.path.join(VERI, "online_retail_2010_2011.csv")

if os.path.exists(CSV):
    df = pd.read_csv(CSV, dtype={"Invoice": str, "StockCode": str},
                     parse_dates=["InvoiceDate"])
    print("Hızlı yol: CSV okundu.")
elif os.path.exists(XLSX):
    df = pd.read_excel(XLSX, sheet_name="Year 2010-2011",
                       dtype={"Invoice": str, "StockCode": str})
    df.to_csv(CSV, index=False)
    print("İlk açılış: Excel okundu ve CSV olarak kaydedildi.")
else:
    print("Veri dosyası bulunamadı:", os.path.abspath(XLSX))
    print("Veri/online_retail/BENIOKU.md dosyasındaki adımları izle.")

print("Satır, sütun:", df.shape)

Aşağıdaki hücre bugün kullanacağımız iki yardımcı fonksiyonu tanımlar. **İçini şimdilik okumana gerek yok**; Bölüm 3'te ne işe yaradıklarını göreceksin. Yalnızca çalıştır.

In [ ]:
def adim(veri, maske, aciklama):
    """Maskesi True olan satırları tutar; kaç satır çıktığını yazar."""
    sonra = veri[maske]
    print(f"{aciklama:<30} çıkan: {len(veri) - len(sonra):>7,}   kalan: {len(sonra):>7,}")
    return sonra


def kontrol_temiz(veri):
    """Temizlenmiş tabloyu sınıf kararlarına göre denetler."""
    testler = {
        "İptal ('C') fatura yok":        not veri["Invoice"].str.startswith("C").any(),
        "Düzeltme ('A') faturası yok":   not veri["Invoice"].str.startswith("A").any(),
        "Bütün adetler > 0":             (veri["Quantity"] > 0).all(),
        "Bütün fiyatlar > 0":            (veri["Price"] > 0).all(),
        "Ürün dışı kod yok":             veri["StockCode"].str[0].str.isdigit().all(),
        "Tekrarlı satır yok":            not veri.duplicated().any(),
    }
    for ad, sonuc in testler.items():
        print("✅" if sonuc else "❌", ad)
    if all(testler.values()):
        print(f"\nTemiz tablo hazır: {len(veri):,} satır.")

---
## Bölüm 1: Pandas ile veriyi tanımak (35 dk)
Geçen hafta Python'u 5 elemanlı bir sepet listesiyle hatırladık. Şimdi **yarım milyondan fazla** satırlık gerçek veride aynı soruları soracağız.

| Geçen hafta (liste) | Bu hafta (pandas) |
|---|---|
| `len(sepet)` | `df.shape` |
| `sepet[0:3]` | `df.head(3)` |
| `for` + `if` ile süzmek | maske: `df[df["Quantity"] < 0]` |
| sözlükle saymak | `.value_counts()`, `.groupby()` |

### 1.1 İlk bakış

In [ ]:
df.head()

**🔮 Tahmin et:** Tablonun kaç satırı ve kaç sütunu var? Önce kaba bir tahmin yaz.

*Tahminim:* ...

In [ ]:
df.shape

In [ ]:
df.info()

**✍️ Tabloyu doldur:** Her sütun ne tür bir değişken? (Seviye testinin ilk sorusunu hatırla: 'Ülke' sütunu hangi türdü?)

| Sütun | Anlamı | Tür (sayısal / kategorik / metin / zaman) | Ölçek (nominal / ordinal / aralık / oran) |
|---|---|---|---|
| Invoice | | | |
| StockCode | | | |
| Description | | | |
| Quantity | | | |
| InvoiceDate | | | |
| Price | | | |
| Customer ID | | | |
| Country | | | |

**🔮 Düşün:** `Customer ID` sayı olarak (`float64`) okunmuş. Müşteri numarasının ortalamasını almak anlamlı mı? Bu sütun gerçekte hangi türdür?

### 1.2 Sütun seçmek

**🔮 Tahmin et:** Aşağıdaki iki satırın çıktısı aynı mı olur? Tek köşeli parantez ile çift köşeli parantez arasında ne fark var?

*Tahminim:* ...

In [ ]:
print(type(df["Country"]))
print(type(df[["Country"]]))

**Series ve DataFrame.** Pandas'ın iki temel yapısı:

| | Series | DataFrame |
|---|---|---|
| Ne? | **Tek bir sütun** | **Tablo**: aynı satır etiketlerini paylaşan sütunlar |
| Boyut | 1 boyutlu: `(satır,)` | 2 boyutlu: `(satır, sütun)` |
| Geçen haftaki karşılığı | **sözlük**: etiket → değer | etiketleri ortak olan sözlüklerin yan yana dizilmesi |
| Nasıl seçilir | `df["Country"]` | `df[["Country"]]`, `df[["Country", "Quantity"]]` |

İkisinde de en solda bir **index** vardır: satırların **etiketi**. Index veri değildir; satırı bulmak için kullandığımız addır (sözlükteki anahtar gibi).

**🔮 Tahmin et:** İki satırın `shape` çıktısı ne olur? İkisinde de kaç sayı yazar?

*Tahminim:* ...

In [ ]:
print(df["Country"].shape)
print(df[["Country"]].shape)

Bir Series'i yakından görelim:

In [ ]:
df["Quantity"].head(3)

**🔮 Düşün:** Çıktıda soldaki `0, 1, 2` nedir, sağdaki sayılar nedir? En alttaki `Name:` ve `dtype:` neyi anlatıyor?

*Cevabım:* ...

**✍️ Kendin yaz:** `Country` ve `Quantity` sütunlarından oluşan bir **tablo** seç ve ilk 5 satırını göster. Sonra türünü yazdır.

In [ ]:
# Kodunu buraya yaz

<details><summary>💡 İpucu 1</summary>

Birden çok sütun seçmek için sütun adlarını bir **liste** olarak ver: `df[["A", "B"]]`. Dıştaki köşeli parantez seçim, içteki liste.

</details>

**🐞 Hata avı:** Müşteri numarası sütununu seçmek istiyoruz.

Hücreyi çalıştır, hata mesajının **son satırını** oku: hatanın türü ne, hangi satırı gösteriyor? Sonra alttaki hücrede düzelt.

In [ ]:
df["CustomerID"].head()

In [ ]:
# Düzeltilmiş kod

<details><summary>💡 İpucu 1</summary>

`df.columns` yazıp sütun adlarını birebir gör. Boşluklara dikkat.

</details>

### 1.3 Özet istatistikler

In [ ]:
df.describe()

**🔮 Tahmin et:** `Quantity` için ortalama mı büyük olur, medyan mı? (Testteki '100, 120, 110, 90, 5000' sorusunu hatırla.)

*Tahminim:* ...

In [ ]:
print("Ortalama:", df["Quantity"].mean())
print("Medyan  :", df["Quantity"].median())

Ortalamayı yukarı ve aşağı çeken satırlara bakalım:

In [ ]:
df.nlargest(3, "Quantity")

In [ ]:
df.nsmallest(3, "Quantity")

**🔮 Düşün:** En büyük ve en küçük satırlar arasında bir ilişki görüyor musun? Ürün adlarına ve müşteri numaralarına bak. Bu gözlemi **Bölüm 4'te** tekrar hatırlayacağız.

### 1.4 Kategorik sütunlar

In [ ]:
df["Country"].value_counts().head(10)

**✍️ Kendin yaz:** Veride **kaç farklı ülke** ve **kaç farklı ürün kodu (`StockCode`)** var?

In [ ]:
# Kodunu buraya yaz

<details><summary>💡 İpucu 1</summary>

`.value_counts()` her değeri sayar. Kaç *farklı* değer olduğunu veren yöntemin adı `n` ile başlar.

</details>

<details><summary>💡 İpucu 2</summary>

`df["Country"].nunique()`

</details>

**`value_counts()` ne döndürür?** Sonucu bir değişkene alıp bakalım.

**🔮 Tahmin et:** `ulkeler` bir Series mi, DataFrame mi? Index'inde (solda) ne yazar, değerlerinde (sağda) ne?

*Tahminim:* ...

In [ ]:
ulkeler = df["Country"].value_counts()
print(type(ulkeler))
ulkeler.head(3)

Index'te ülke adları, değerlerde sayılar var. Yani `ulkeler` tam bir **sözlük** gibi: geçen hafta `adetler["WHITE METAL LANTERN"]` yazdığımız gibi, burada da **etiketle** okuyabiliriz. Sıradaki yerine göre okumak için `.index[...]` ve `.iloc[...]` var.

**🔮 Tahmin et:** Aşağıdaki üç satır ne yazdırır? (`[2]` kaçıncı sıra?)

*Tahminim:* ...

In [ ]:
print(ulkeler["Germany"])   # etiketle oku (sözlük gibi)
print(ulkeler.index[2])     # 3. sıradaki etiket
print(ulkeler.iloc[2])      # 3. sıradaki değer

**✍️ Kendin yaz:** Fransa'dan kaç satır var? Listede **5. sıradaki** ülke hangisi?

In [ ]:
# Kodunu buraya yaz

<details><summary>💡 İpucu 1</summary>

Sayma 0'dan başlar: 5. sıra `[4]`.

</details>

### 1.5 Eksik değerler

In [ ]:
df.isnull().sum()

**🔮 Tahmin et:** `Customer ID` satırlarının yüzde kaçı boş? Önce tahmin et. (İpucu: `True` sayısal hesapta 1 sayılır.)

*Tahminim:* ...

In [ ]:
print(df["Customer ID"].isnull().mean() * 100)

**🔮 Düşün:** Müşteri numarası boş olan bir satır yine de **gerçek bir satış** olabilir mi? "Herkese en çok satanı öner" dersek bu satırlara ihtiyacımız var mı? "Bu müşteriye öner" dersek?

#### ⭐ Ek alıştırmalar (Bölüm 1): erken bitirenler ve ev için
Her biri bir ya da iki satır. Önce cevabı tahmin et, sonra yaz.

**E1.** Almanya'dan kaç satır var? (Yukarıdaki `ulkeler` işini görür.)

In [ ]:
# E1

**E2.** Fiyatı (`Price`) en yüksek 3 satırı göster. Bunlar gerçekten ürün mü? `Invoice` sütununa da bak.

In [ ]:
# E2

**E3.** `Description` sütunu boş olan satırlarda `Price` ne oluyor? (İpucu: `df.loc[maske, "Price"].describe()`; maskeyi `.isnull()` ile kur.)

In [ ]:
# E3

---
## Bölüm 2: Verinin tuhaflıkları (25 dk)
Gerçek veri temiz gelmez. Bu bölümde **henüz temizlik yapmıyoruz**: tuhaf satırları buluyor ve ne olduklarını anlamaya çalışıyoruz. Kararı bölüm sonunda vereceğiz.

### 2.1 'C' ile başlayan faturalar

In [ ]:
iptal = df["Invoice"].str.startswith("C")
print(iptal.sum(), "satır")
df[iptal].head()

**Maske nedir?** `iptal` değişkeni bir **Series**: her satır için tek bir değer, `True` ya da `False`. Biz buna **maske** diyoruz. `df[iptal]` tablodan yalnızca maskesi `True` olan satırları alır.

**🔮 Tahmin et:** `iptal.head()` ne gösterir? `dtype` ne yazar?

*Tahminim:* ...

In [ ]:
print(iptal.dtype)
iptal.head()

`True` hesapta **1**, `False` **0** sayılır. Bu yüzden `iptal.sum()` kaç tane `True` olduğunu, `iptal.mean()` ise oranını verir.

**✍️ Kendin yaz:** Satırların yüzde kaçı iptal?

In [ ]:
# Kodunu buraya yaz

**`.loc` ile satır ve sütunu birlikte seçmek:** `df.loc[satırlar, sütunlar]`. Önce hangi satırlar (maske), virgülden sonra hangi sütun(lar).

| Yazım | Ne verir? |
|---|---|
| `df[iptal]` | İptal satırları, **bütün** sütunlar (DataFrame) |
| `df.loc[iptal, "Quantity"]` | İptal satırları, **yalnızca** `Quantity` (Series) |
| `df.loc[iptal, ["Invoice", "Quantity"]]` | İptal satırları, iki sütun (DataFrame) |

**🔮 Tahmin et:** İptal satırlarında `Quantity` hep eksi mi? Önce tahmin et.

*Tahminim:* ...

In [ ]:
df.loc[iptal, "Quantity"].describe()

### 2.2 Eksi adet ama iptal değil
**🐞 Hata avı (sessiz hata):** İptal olmayan ama adedi eksi olan satırları arıyoruz. Bu kez **hata mesajı çıkmayacak**: kod çalışır ama sonuç yanlıştır. Çıkan satır sayısını, iptal satırı sayısıyla (2.1) karşılaştır. Bir terslik var mı? Alttaki hücrede düzelt.

In [ ]:
df[df["Quantity"] < 0 & ~iptal].shape

In [ ]:
# Düzeltilmiş kod

<details><summary>💡 İpucu 1</summary>

Python önce `&` işlemini yapar, sonra `<` karşılaştırmasını. Yani kod aslında `df["Quantity"] < (0 & ~iptal)` olarak okunuyor.

</details>

<details><summary>💡 İpucu 2</summary>

Her koşulu kendi parantezine al: `(koşul1) & (koşul2)`

</details>

In [ ]:
# İlk 10 satırı göster

**🔮 Düşün:** Bu satırların `Price` ve `Customer ID` değerlerinde ortak bir şey var mı? Sence bunlar satış mı, depo kaydı mı?

### 2.3 Sıfır ya da eksi fiyat
**✍️ Kendin yaz:** `Price` değeri 0 ya da daha küçük olan satırları bul. Kaç tane? En sık görülen `Description` değerleri neler?

In [ ]:
# Kodunu buraya yaz

<details><summary>💡 İpucu 1</summary>

Önce bir maske kur: `fiyat_sorunlu = df["Price"] <= 0`. Kaç tane olduğunu `.sum()` söyler.

</details>

<details><summary>💡 İpucu 2</summary>

Maskeyi satır seçmek için kullan ve yalnızca `Description` sütununu al: `df.loc[maske, "Description"]`. Sonra `.value_counts()`.

</details>

In [ ]:
df[df["Price"] < 0]

**✍️ Kendin yaz:** Fatura numaralarının **ilk karakterine** göre kaç satır olduğunu say. 'C' ve 'A' dışında başka harf var mı? (İpucu: `.str[0]` ve `.value_counts()`)

In [ ]:
# Kodunu buraya yaz

### 2.4 Ürün olmayan kodlar

In [ ]:
# Ürün kodları genelde 5 rakamla başlar. Rakamla başlamayanlara bakalım:
urun_disi = ~df["StockCode"].str[0].str.isdigit()
df.loc[urun_disi, ["StockCode", "Description"]].value_counts().head(10)

**🔮 Düşün:** `POST`, `DOT`, `M`, `BANK CHARGES` birer ürün mü? Bunları bir müşteriye "öneri" olarak göstermek nasıl görünürdü?

### 2.5 Tekrar eden satırlar

**🔮 Tahmin et:** Tamamen aynı olan (bütün sütunları aynı) kaç satır vardır: 0 mı, birkaç tane mi, binlerce mi?

*Tahminim:* ...

In [ ]:
print(df.duplicated().sum())

### 2.6 Karar tablosu (grup çalışması, 5 dk)
Aşağıdaki tabloyu grubunla doldur. **Tek doğru cevap yok**, önemli olan gerekçen. Amaç: *herkese en çok satan ürünleri önermek.*

| Satır türü | Tut / Çıkar | Gerekçe |
|---|---|---|
| Müşteri numarası boş | | |
| Fatura 'C' ile başlıyor (iptal) | | |
| Fatura 'A' ile başlıyor | | |
| İptal değil ama adet eksi | | |
| Fiyat 0 ya da eksi | | |
| Ürün dışı kodlar (POST, M, ...) | | |
| Tamamen aynı (tekrarlı) satırlar | | |

Gruplar kararlarını söyledikten sonra **sınıfın ortak kararını** aşağıya yaz. Gelecek haftalarda hepimiz aynı temiz tabloyla çalışacağız.

*Sınıf kararı:* ...

---
## Bölüm 3: Kararı koda dökmek (20 dk)
Her karar bir **maske** demek: tutmak istediğimiz satırlar `True`. `adim()` fonksiyonu maskeyi uygular ve kaç satır çıktığını yazar. Böylece bir **temizlik günlüğü** oluşur.

İlk adım hazır. Sonraki adımlarda kodun bir kısmı boş (`___`); her adımda biraz daha fazlasını sen yazacaksın.

In [ ]:
d = df.copy()   # ham veriyi bozmamak için kopya üzerinde çalışıyoruz

# Adım 1 (hazır): iptal faturaları çıkar
d = adim(d, ~d["Invoice"].str.startswith("C"), "İptal faturalar")

**🔧 Değiştir:** Adım 2: 'A' ile başlayan düzeltme faturalarını çıkar. Yalnızca `___` yerini doldur.

In [ ]:
d = adim(d, ~d["Invoice"].str.startswith(___), "'A' faturalar")

**🔧 Değiştir:** Adım 3: adedi sıfır ya da eksi olanları çıkar. Tutmak istediğimiz satırlar hangileri?

In [ ]:
d = adim(d, d["Quantity"] ___ 0, "Adet sıfır ya da eksi")

**✍️ Kendin yaz:** Adım 4 ve 5: fiyatı sıfır ya da eksi olanları ve ürün dışı kodları çıkar. Her biri tek satır, `adim(...)` kalıbını kullan.

In [ ]:
# Adım 4: fiyat

# Adım 5: ürün dışı kodlar

<details><summary>💡 İpucu 1</summary>

Adım 3'ün kalıbını kopyalamadan, kendin yeniden yaz: `d = adim(d, MASKE, "açıklama")`.

</details>

<details><summary>💡 İpucu 2</summary>

Ürün dışı kodlar için 2.4'teki maskeyi hatırla. Ama bu kez **`d` üzerinden** kur ve tutmak istediğimiz satırlar `True` olsun: rakamla başlayanlar.

</details>

**🐞 Hata avı (sessiz hata):** Adım 6 tekrarlı satırları çıkarmalı. Deneme amaçlı sonucu `d_deneme` adına yazdık. Kod çalışıyor ama günlükteki **kalan** sayısına bak: bir önceki adımdan daha mı az, daha mı çok? Neden? Alttaki hücrede düzelt.

In [ ]:
d_deneme = adim(df, ~df.duplicated(), "Tekrarlı satırlar")

In [ ]:
# Düzeltilmiş kod

<details><summary>💡 İpucu 1</summary>

Fonksiyona hangi tabloyu verdin: temizlenmekte olan `d` mi, ham `df` mi?

</details>

Şimdi temiz tabloyu adlandır ve denetle:

In [ ]:
df_temiz = d
kontrol_temiz(df_temiz)

**🔮 Düşün:** Temizlikten sonra `Customer ID` hâlâ boş olan satırlar var. Bugünkü amacımız (herkese en çok satanı önermek) için bu doğru bir karar mı? Gelecek haftalarda **kişiye özel** öneri yaparken ne değişecek?

In [ ]:
print(df_temiz["Customer ID"].isnull().sum(), "satırda müşteri numarası yok")

### Temiz tabloyu kaydet
Gelecek hafta sepet analizine bu temiz tabloyla başlayacağız. Temizliği her seferinde baştan çalıştırmamak için kaydedelim.

In [ ]:
TEMIZ = os.path.join(VERI, "online_retail_temiz.csv")
df_temiz.to_csv(TEMIZ, index=False)
print("Kaydedildi:", TEMIZ, df_temiz.shape)

**🔮 Düşün:** `index=False` ne işe yarıyor? (Index'in veri değil, satır etiketi olduğunu hatırla.) Yazmasaydık dosyada ne fazladan olurdu?

#### ⭐ Ek alıştırmalar (Bölüm 3)
**E4.** Temiz tabloda fiyatı en yüksek 3 satır hangileri? E2'deki sonuçla karşılaştır: temizlik ne değiştirdi? Yeni listede hâlâ şüpheli görünen bir satır var mı?

In [ ]:
# E4

**E5.** Hangi **ayda** en çok fatura kesilmiş? (İpucu: `df_temiz["InvoiceDate"].dt.month` her satırın ay numarasını verir; `groupby`'a sütun adı yerine bunu verebilirsin.)

🕵️ Sonra veri hangi tarihler arasında, kontrol et (`.min()`, `.max()`). Aralık (12. ay) sonucuna güvenebilir miyiz?

In [ ]:
# E5

---
## Bölüm 4: İlk öneri sistemi: en çok satanlar (30 dk)
En basit öneri sistemi herkese aynı listeyi gösterir: **en popüler ürünler.** Kişiselleştirilmemiştir ama yeni gelen, hakkında hiçbir şey bilmediğimiz bir müşteri için bile çalışır.

Peki "en popüler" ne demek? Önce küçük bir örnekte düşünelim.

### 4.1 Önce küçük veri

In [ ]:
mini = pd.DataFrame({
    "Fatura": ["1", "1", "2", "2", "2", "3"],
    "Urun":   ["Çay", "Süt", "Çay", "Ekmek", "Süt", "Çay"],
    "Adet":   [2, 1, 1, 30, 2, 5],
})
mini

Geçen haftaki gibi, sözlük ve döngüyle sayalım:

**🔮 Tahmin et:** Aşağıdaki döngü bittiğinde `sayac` sözlüğünde ne yazar?

*Tahminim:* ...

In [ ]:
sayac = {}
for urun in mini["Urun"]:
    if urun in sayac:
        sayac[urun] += 1
    else:
        sayac[urun] = 1
print(sayac)

Pandas aynı işi tek satırda yapar:

In [ ]:
mini["Urun"].value_counts()

`groupby` ise bir adım daha ileri gider: tabloyu bir sütuna göre **gruplara böler**, her gruba bir **hesap uygular** ve sonuçları **birleştirir**.

**🔮 Tahmin et:** Elle hesapla: her ürünün **toplam adedi** ne?

*Tahminim:* ...

In [ ]:
mini.groupby("Urun")["Adet"].sum()

**🔮 Tahmin et:** Elle hesapla: her ürün **kaç farklı faturada** geçiyor?

*Tahminim:* ...

In [ ]:
mini.groupby("Urun")["Fatura"].nunique()

**✍️ Kendin yaz:** Şimdi **grup anahtarını değiştir**: her faturada toplam kaç ürün (adet) var? Yani her **sepetin büyüklüğü** ne?

In [ ]:
# Kodunu buraya yaz

<details><summary>💡 İpucu 1</summary>

Bu kez ürüne göre değil, **faturaya** göre grupluyoruz. Yukarıdaki satırda yalnızca iki kelime değişir.

</details>

**🔮 Düşün:** Bu sonuç bir öneri listesi değil. Her satırı bir **sepet**. Aynı tabloya ürün açısından da sepet açısından da bakabiliyoruz. Dersin sonunda sepetlere geri döneceğiz.

**🔮 Düşün:** Bu küçük veride "en popüler ürün" hangisi: Ekmek mi, Çay mı? İki ölçü neden farklı cevap veriyor? Hangisi bir mağazanın ana sayfasında gösterilmeye daha uygun?

### 4.2 Gerçek veride: toplam adede göre ilk 10
**🧩 Sırala:** Aşağıdaki parçalar karışık verildi. Doğru sıraya koyup **tek bir satır** olarak yaz: temiz tabloda **toplam adedi en yüksek 10 ürünü** bulsun. Sonucu `top_adet` adında bir değişkene ata.

```
.nlargest(10)
["Quantity"]
df_temiz.groupby("Description")
.sum()
```

In [ ]:
top_adet = 
top_adet

<details><summary>💡 İpucu 1</summary>

`mini` örneğindeki sıra: önce **neye göre grupla**, sonra **hangi sütun**, sonra **hangi hesap**. En sonda sıralayıp ilk 10'u al.

</details>

### 4.3 🕵️ İz sür: birinci sıradaki ürün
Listenin başındaki ürün gerçekten bu mağazanın en çok satan ürünü mü? İz sürelim.

**✍️ Kendin yaz:** Birinci sıradaki ürünün adını `urun` değişkenine al. Sonra **ham veride (`df`)** bu ürünün geçtiği bütün satırları göster.

In [ ]:
# Kodunu buraya yaz

<details><summary>💡 İpucu 1</summary>

Bir Series'in ilk etiketini `.index[0]` verir.

</details>

<details><summary>💡 İpucu 2</summary>

Ham tabloda süzmek için: `df[df["Description"] == urun]`

</details>

**🔮 Düşün:**
1. Kaç satır var? Faturaların numaralarına, saatlerine ve müşteri numarasına bak. Ne olmuş?
2. Temizlikte iptal faturasını çıkardık. Bu ürün neden yine de listenin başında?
3. 1.3'te `nlargest` ile gördüğün satırları hatırla. İkinci sıradaki ürün için de aynısı geçerli mi? (İstersen `top_adet.index[1]` ile bak.)

*Cevabım:* ...

### 4.4 Hangi ölçü?
**🔧 Değiştir:** 4.2'deki satırı iki kez uyarla:
- `top_fatura`: ürünün geçtiği **farklı fatura sayısına** göre ilk 10
- `top_musteri`: ürünü alan **farklı müşteri sayısına** göre ilk 10

In [ ]:
# top_fatura

# top_musteri

<details><summary>💡 İpucu 1</summary>

4.1'de `mini` için "kaç farklı faturada" sorusunu hangi satırla cevapladın?

</details>

İki listede kaç ortak ürün var? Geçen hafta öğrendiğin **küme** (`set`) işini görür:

**🔮 Tahmin et:** `top_adet` ile `top_fatura` listelerinin kaç ürünü ortaktır: 10'a yakın mı, yarısı mı, daha az mı?

*Tahminim:* ...

In [ ]:
ortak = set(top_adet.index) & set(top_fatura.index)
print(len(ortak), "ortak ürün")
ortak

**✍️ Karar ver (gerekçeli):** Mağazanın ana sayfasında **herkese** gösterilecek 10 ürünü hangi ölçüyle seçerdin: adet, fatura sayısı, müşteri sayısı ya da başka bir şey? Neden? Ölçünün zayıf yanı ne?

*Kararım ve gerekçem:* ...

### 4.5 Ülkeye göre en çok satanlar
Müşterinin yalnızca ülkesini biliyorsak listeyi biraz kişiselleştirebiliriz.

**🔧 Değiştir:** Önce yalnızca **Almanya** (`"Germany"`) satırlarını al, sonra fatura sayısına göre ilk 5 ürünü bul.

In [ ]:
# Kodunu buraya yaz

**✍️ Kendin yaz:** Bunu bir **fonksiyona** çevir: `en_cok_satan(veri, ulke, n)` verilen ülke için fatura sayısına göre ilk `n` ürünü döndürsün. Sonra Almanya ve Fransa için dene.

In [ ]:
def en_cok_satan(veri, ulke, n=5):
    # Kodunu buraya yaz
    pass


en_cok_satan(df_temiz, "France")

<details><summary>💡 İpucu 1</summary>

Fonksiyonun gövdesi, bir üstteki hücrede yazdığın iki satırdır. `"Germany"` yerine `ulke` yaz ve sonucu `return` ile döndür.

</details>

**🔮 Tahmin et:** Veride hiç satış olmayan bir ülke için ne olur? Hata mı verir, boş liste mi döner?

*Tahminim:* ...

In [ ]:
en_cok_satan(df_temiz, "Turkey")

**🔧 Değiştir (soğuk başlangıç):** Ülke için hiç veri yoksa fonksiyon boş liste yerine **bütün mağazanın** en çok satanlarını döndürsün. Fonksiyonunu güncelle ve `"Turkey"` ile tekrar dene.

In [ ]:
# Kodunu buraya yaz

<details><summary>💡 İpucu 1</summary>

`len(secim) == 0` ise `secim` yerine bütün `veri` ile devam et. Geçen haftaki `if` bloğunu hatırla.

</details>

#### ⭐ Ek alıştırmalar (Bölüm 4)
**E6.** Bir müşteri ortalama kaç farklı fatura kesmiş? (Müşteriye göre grupla, faturaları say, ortalamasını al.)

In [ ]:
# E6

**E7.** Her ülkede kaç farklı müşteri var? En çok müşterisi olan 5 ülke? İngiltere dışındaki ülkeler için "ülkeye göre en çok satan" listesine ne kadar güvenebiliriz?

In [ ]:
# E7

**E8.** Bir sepette (faturada) ortalama kaç **farklı ürün** var? Ortalama ile medyanı karşılaştır. (Gelecek haftanın sepet analizi bu sayıya çok bağlı.)

In [ ]:
# E8

---
### Köprü: listenin bilmediği şey
Bir müşterinin sepetinde şu an **REGENCY CAKESTAND 3 TIER** var. Bugünkü listemiz ona ne önerir?

**🔮 Düşün:** Liste, müşterinin sepetinde ne olduğunu hiç kullanıyor mu? "Bu ürünü alanlar şunları da aldı" diyebilmek için hangi bilgiye ihtiyacımız var: satır mı, fatura mı, müşteri mi?

Bu sorunun cevabı **sepet analizi** (birliktelik kuralları). Mantığını şimdi konuşacağız, kodunu gelecek hafta yazacağız.

---
## Kapanış
**Bugün:** Veriyi tanıdık, tuhaf satırları bulduk, sınıf olarak temizlik kararı verip koda döktük ve ilk öneri sistemimizi kurduk: *en çok satanlar*. "Popüler" kelimesinin tek bir anlamı olmadığını, ölçü seçiminin sonucu değiştirdiğini gördük.

**Gelecek hafta:** Fatura başına sepetler kurup "bunu alan şunu da aldı" kurallarını çıkaracağız.

**Haftaya kadar:**
1. Bu defterde boş bıraktığın ✍️, 🔧 ve 🐞 hücrelerini tamamla.
2. 4.4'teki ölçü kararını ve gerekçeni yaz.
3. Kendi seçtiğin bir ülke için `en_cok_satan` sonucunu Almanya ile karşılaştır: ortak ürün var mı?
4. Aşağıdaki **🧠 Kendini sına** sorularını cevapla. Vaktin olursa ⭐ ek alıştırmaları (E1–E8) yap.

---
### 🧠 Kendini sına (5 soru)
Önce kodu çalıştırmadan cevabını **yaz**. Sonra alttaki hücreyi çalıştırıp kontrol et.

1. `s = pd.Series([3, 1, 2], index=["a", "b", "c"])` için `s["b"]` ve `s.index[0]` ne verir?
2. `df["Country"].shape` ile `df[["Country"]].shape` arasındaki fark ne?
3. `m = pd.Series([True, False, True, True])` için `m.sum()` ve `m.mean()` ne verir?
4. `mini.groupby("Fatura")["Urun"].nunique()` ne verir?
5. `df[df["Price"] > 0 & df["Quantity"] > 0]` çalışmaz. Neden? Nasıl düzeltilir?

*Cevaplarım:*
1. ...
2. ...
3. ...
4. ...
5. ...

In [ ]:
s = pd.Series([3, 1, 2], index=["a", "b", "c"])
print(1, s["b"], s.index[0])
print(2, df["Country"].shape, df[["Country"]].shape)
m = pd.Series([True, False, True, True])
print(3, m.sum(), m.mean())
print(4, mini.groupby("Fatura")["Urun"].nunique().to_dict())
print(5, df[(df["Price"] > 0) & (df["Quantity"] > 0)].shape)